In [16]:
import os
import sqlite3
import pandas as pd
import numpy as np

In [17]:
print("1. Caricamento dataset...")

try:
    df = pd.read_csv('spotify.csv', sep=';', encoding='utf-8', on_bad_lines='skip', low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv('spotify.csv', sep=';', encoding='latin1', on_bad_lines='skip', low_memory=False)

# Pulizia di eventuali spazi nei nomi delle colonne
df.columns = df.columns.str.strip()
print("Colonne caricate:", list(df.columns))

1. Caricamento dataset...
Colonne caricate: ['track_id', 'track_name', 'track_number', 'track_popularity', 'explicit', 'artist_name', 'artist_name & track name', 'artist_popularity', 'artist_followers', 'artist_genres', 'album_id', 'album_name', 'album_release_date', 'anno', 'decade', 'album_total_tracks', 'album_type', 'track_duration_min']


In [18]:
print("2. Pulizia dati e parsing delle date...")

# Rimozione righe nulle sulle chiavi primarie
df_clean = df.dropna(subset=['artist_name', 'album_id', 'album_release_date', 'track_id']).copy()

# Conversione Date ISO-8601 e calcolo anno/decade
df_clean['album_release_date'] = pd.to_datetime(df_clean['album_release_date'], errors='coerce')
df_clean = df_clean.dropna(subset=['album_release_date'])

df_clean['release_year'] = df_clean['album_release_date'].dt.year
df_clean['release_decade'] = (df_clean['release_year'] // 10) * 10
df_clean['album_release_date'] = df_clean['album_release_date'].dt.strftime('%Y-%m-%d')

# Conversione pulita di artist_followers (rimozione decimali e non-cifre)
df_clean['artist_followers'] = (
    df_clean['artist_followers']
    .astype(str)
    .str.split('.').str[0]
    .str.replace(r'\D', '', regex=True)
)
df_clean['artist_followers'] = pd.to_numeric(df_clean['artist_followers'], errors='coerce').fillna(0).astype(int)

# Conversione explicit
df_clean['explicit'] = pd.to_numeric(df_clean['explicit'], errors='coerce').fillna(0).astype(int)

print(f"Dataset pulito. Righe rimanenti: {len(df_clean)}")

2. Pulizia dati e parsing delle date...
Dataset pulito. Righe rimanenti: 8568


In [19]:
print("3. Creazione Star Schema...")

# A. Dimensione Artista
df_clean['artist_id'] = df_clean['artist_name'].astype('category').cat.codes + 1
dim_artist = df_clean[['artist_id', 'artist_name', 'artist_popularity', 'artist_followers']].drop_duplicates(subset=['artist_id'])

# B. Dimensione Album
dim_album = df_clean[['album_id', 'album_name', 'album_release_date', 'album_total_tracks', 'album_type']].drop_duplicates(subset=['album_id'])

# C. Fact Track
fact_track = df_clean[[
    'track_id', 'track_name', 'artist_id', 'album_id', 
    'track_number', 'track_popularity', 'track_duration_min', 
    'explicit', 'release_year', 'release_decade'
]].drop_duplicates(subset=['track_id'])

# D. Dimensione Genere e Tabella Bridge
genres_series = df_clean[['artist_id', 'artist_genres']].dropna().drop_duplicates()
genres_exploded = genres_series.assign(genre_name=genres_series['artist_genres'].astype(str).str.split(',')).explode('genre_name')
genres_exploded['genre_name'] = genres_exploded['genre_name'].str.strip().str.lower()
genres_exploded = genres_exploded[genres_exploded['genre_name'] != '']

dim_genre = pd.DataFrame({'genre_name': genres_exploded['genre_name'].unique()}).reset_index(drop=True)
dim_genre['genre_id'] = dim_genre.index + 1

bridge_artist_genre = genres_exploded.merge(dim_genre, on='genre_name')[['artist_id', 'genre_id']].drop_duplicates()

print("Star Schema creato con successo!")

3. Creazione Star Schema...
Star Schema creato con successo!


In [20]:
db_name = 'spotify_database.db'
print(f"4. Scrittura file '{db_name}'...")

conn = sqlite3.connect(db_name)

dim_artist.to_sql('dim_artist', conn, if_exists='replace', index=False)
dim_album.to_sql('dim_album', conn, if_exists='replace', index=False)
dim_genre.to_sql('dim_genre', conn, if_exists='replace', index=False)
bridge_artist_genre.to_sql('bridge_artist_genre', conn, if_exists='replace', index=False)
fact_track.to_sql('fact_track', conn, if_exists='replace', index=False)

print(f"✅ Database creato con successo! Dimensione: {os.path.getsize(db_name) / (1024 * 1024):.2f} MB")

4. Scrittura file 'spotify_database.db'...


✅ Database creato con successo! Dimensione: 1.25 MB


In [21]:
q1 = """
SELECT 
    release_decade,
    COUNT(track_id) AS total_tracks,
    ROUND(AVG(track_popularity), 2) AS avg_track_popularity,
    ROUND(AVG(track_duration_min), 2) AS avg_duration_min,
    ROUND(SUM(explicit) * 100.0 / COUNT(track_id), 2) AS explicit_pct
FROM fact_track
WHERE release_decade IS NOT NULL
GROUP BY release_decade
ORDER BY release_decade ASC;
"""

print("=== 1. ANALISI TEMPORALE PER DECENNIO ===")
pd.read_sql(q1, conn)

=== 1. ANALISI TEMPORALE PER DECENNIO ===


,release_decade,total_tracks,avg_track_popularity,avg_duration_min,explicit_pct
0,1950,7,53.57,2.86,0.00
1,1960,50,52.66,2.97,0.00
2,1970,79,63.48,4.64,2.53
3,1980,98,59.18,4.23,4.08
4,1990,403,51.21,4.09,21.09
5,2000,913,53.28,3.80,14.46
6,2010,3699,52.14,3.63,24.52
7,2020,3319,52.02,3.14,30.43


In [22]:
q2 = """
WITH ArtistRanked AS (
    SELECT 
        g.genre_name,
        a.artist_name,
        a.artist_popularity,
        DENSE_RANK() OVER (
            PARTITION BY g.genre_name 
            ORDER BY a.artist_popularity DESC
        ) AS rank_in_genre
    FROM dim_artist a
    JOIN bridge_artist_genre bag ON a.artist_id = bag.artist_id
    JOIN dim_genre g ON bag.genre_id = g.genre_id
    GROUP BY g.genre_name, a.artist_id
)
SELECT * FROM ArtistRanked
WHERE rank_in_genre <= 3
ORDER BY genre_name ASC, rank_in_genre ASC
LIMIT 15;
"""

print("=== 2. TOP ARTISTI PER GENERE (DENSE_RANK) ===")
pd.read_sql(q2, conn)

=== 2. TOP ARTISTI PER GENERE (DENSE_RANK) ===


,genre_name,artist_name,artist_popularity,rank_in_genre
0,3 step,AthenaDME,22.0,1
1,acid house,Alabama 3,43.0,1
2,acid rock,The Doors,71.0,1
3,acid rock,Jimi Hendrix,66.0,2
4,acid rock,Grateful Dead,64.0,3
5,acid techno,Charlotte de Witte,60.0,1
6,acoustic country,Luke Bryan,74.0,1
7,acoustic country,Jason Aldean,74.0,1
8,acoustic country,Thomas Rhett,73.0,2
9,acoustic country,Jon Pardi,73.0,2


1.2 Struttura e Tipi di dati: ogni colonna ha un .dtype: calcoli, ordinamento, testo, date ...

In [23]:
q3 = """
SELECT 
    t.track_name,
    a.artist_name,
    al.album_name,
    t.track_popularity,
    ROUND(AVG(t.track_popularity) OVER(PARTITION BY t.album_id), 2) AS avg_album_popularity,
    ROUND(t.track_popularity - AVG(t.track_popularity) OVER(PARTITION BY t.album_id), 2) AS diff_from_album_avg
FROM fact_track t
JOIN dim_artist a ON t.artist_id = a.artist_id
JOIN dim_album al ON t.album_id = al.album_id
WHERE t.track_popularity > 0
ORDER BY t.track_popularity DESC
LIMIT 10;
"""

print("=== 3. SCARTO POPOLARITÀ TRACCIA VS MEDIA ALBUM ===")
df_q3 = pd.read_sql(q3, conn)

# Chiusura formale della connessione al database
conn.close()

df_q3

=== 3. SCARTO POPOLARITÀ TRACCIA VS MEDIA ALBUM ===


,track_name,artist_name,album_name,track_popularity,avg_album_popularity,diff_from_album_avg
0,Golden,HUNTR/X,KPop Demon Hunters (Soundtrack from the Netfli...,99.0,91.44,7.56
1,Opalite,Taylor Swift,The Life of a Showgirl,97.0,93.27,3.73
2,Man I Need,Olivia Dean,Man I Need,95.0,95.00,0.00
3,Elizabeth Taylor,Taylor Swift,The Life of a Showgirl,95.0,93.27,1.73
4,Soda Pop,Saja Boys,KPop Demon Hunters (Soundtrack from the Netfli...,94.0,91.44,2.56
5,Father Figure,Taylor Swift,The Life of a Showgirl,94.0,93.27,0.73
6,BIRDS OF A FEATHER,Billie Eilish,HIT ME HARD AND SOFT,94.0,82.20,11.80
7,How It’s Done,HUNTR/X,KPop Demon Hunters (Soundtrack from the Netfli...,93.0,91.44,1.56
8,Your Idol,Saja Boys,KPop Demon Hunters (Soundtrack from the Netfli...,93.0,91.44,1.56
9,The Life of a Showgirl (feat. Sabrina Carpenter),Taylor Swift,The Life of a Showgirl,93.0,93.27,-0.27


In [24]:
import sqlite3
import pandas as pd

# ==========================================
# 1. CARICAMENTO DATI DAL DATABASE SQLITE
# ==========================================
print("1. Estrazione dati dal database per l'analisi statistica...")

conn = sqlite3.connect('spotify_database.db')

query_stats = """
SELECT 
    t.track_popularity,
    t.track_duration_min,
    t.explicit,
    t.release_year,
    a.artist_followers,
    a.artist_popularity
FROM fact_track t
JOIN dim_artist a ON t.artist_id = a.artist_id
WHERE t.track_popularity > 0;
"""

df_stat = pd.read_sql(query_stats, conn)
conn.close()

print(f"✅ Dati estratti con successo: {len(df_stat)} righe caricate.")

1. Estrazione dati dal database per l'analisi statistica...
✅ Dati estratti con successo: 8067 righe caricate.


In [25]:
import statistics

# ==========================================
# 2. STATISTICA DESCRITTIVA (MODULE STATISTICS)
# ==========================================
print("="*50)
print("STATISTICHE DESCRITTIVE (MODULE STATISTICS)")
print("="*50)

pop_list = df_stat['track_popularity'].tolist()

print(f"Media Popolarità:              {statistics.mean(pop_list):.2f}")
print(f"Mediana Popolarità:            {statistics.median(pop_list):.2f}")
print(f"Deviazione Standard:           {statistics.stdev(pop_list):.2f}")
print(f"Varianza:                      {statistics.variance(pop_list):.2f}")

STATISTICHE DESCRITTIVE (MODULE STATISTICS)
Media Popolarità:              55.61
Mediana Popolarità:            60.00
Deviazione Standard:           20.53
Varianza:                      421.55


In [26]:
# ==========================================
# 3. MATRICE DI CORRELAZIONE DI PEARSON
# ==========================================
print("="*50)
print("MATRICE DI CORRELAZIONE DI PEARSON")
print("="*50)

corr_matrix = df_stat.corr()
print(corr_matrix.round(3))

MATRICE DI CORRELAZIONE DI PEARSON
                    track_popularity  track_duration_min  explicit  \
track_popularity               1.000               0.099     0.130   
track_duration_min             0.099               1.000     0.012   
explicit                       0.130               0.012     1.000   
release_year                  -0.021              -0.267     0.139   
artist_followers               0.207               0.175     0.102   
artist_popularity              0.461               0.209     0.175   

                    release_year  artist_followers  artist_popularity  
track_popularity          -0.021             0.207              0.461  
track_duration_min        -0.267             0.175              0.209  
explicit                   0.139             0.102              0.175  
release_year               1.000             0.088             -0.068  
artist_followers           0.088             1.000              0.644  
artist_popularity         -0.068          

In [27]:
import numpy as np

# ==========================================
# 4. REGRESSIONE LINEARE OLS BIVARIATA (NumPy)
# ==========================================
print("="*50)
print("REGRESSIONE LINEARE OLS BIVARIATA (NumPy)")
print("="*50)

# Modello: track_popularity = alpha + beta * artist_popularity
x = df_stat['artist_popularity'].to_numpy()
y = df_stat['track_popularity'].to_numpy()

# Calcolo analitico dei coefficienti OLS via Covarianza e Varianza: Beta = Cov(X,Y) / Var(X)
beta_biv = np.cov(x, y)[0, 1] / np.var(x, ddof=1)
alpha_biv = np.mean(y) - beta_biv * np.mean(x)

# Calcolo R-squared
r_matrix = np.corrcoef(x, y)
r_squared_biv = r_matrix[0, 1]**2

print(f"Variabile Indipendente (X): artist_popularity")
print(f"Variabile Dipendente (Y)  : track_popularity")
print(f"Coeff. Angolare (Beta)    : {beta_biv:.4f}")
print(f"Intercetta (Alpha)        : {alpha_biv:.4f}")
print(f"R-squared (R²)            : {r_squared_biv:.4f}")

REGRESSIONE LINEARE OLS BIVARIATA (NumPy)
Variabile Indipendente (X): artist_popularity
Variabile Dipendente (Y)  : track_popularity
Coeff. Angolare (Beta)    : 0.4974
Intercetta (Alpha)        : 20.4532
R-squared (R²)            : 0.2124


In [28]:
import numpy as np
import pandas as pd

# ==========================================
# 5. REGRESSIONE LINEARE OLS MULTIPLA (NumPy Algebra)
# ==========================================
print("="*50)
print("REGRESSIONE LINEARE OLS MULTIPLA (NumPy Algebra)")
print("="*50)

# Trasformazione logaritmica sui follower per ridurre la skewness (+1 per evitare log(0))
df_stat['log_artist_followers'] = np.log1p(df_stat['artist_followers'])

# Matrice delle covariate X (con colonna di 1 per l'intercetta)
X = df_stat[['explicit', 'log_artist_followers', 'release_year', 'track_duration_min']].copy()
X.insert(0, 'Intercept', 1.0)
Y = df_stat['track_popularity'].to_numpy()

# Stima dei coefficienti OLS tramite equazioni normali: Beta = (X^T * X)^(-1) * X^T * Y
beta_mult, residuals, rank, s = np.linalg.lstsq(X.values, Y, rcond=None)

# Calcolo R-squared
ss_tot = np.sum((Y - np.mean(Y))**2)
ss_res = residuals[0] if len(residuals) > 0 else np.sum((Y - np.dot(X.values, beta_mult))**2)
r_squared_mult = 1 - (ss_res / ss_tot)

coefficients = pd.DataFrame({
    'Variable': X.columns,
    'Coefficient (Beta)': beta_mult
})

print(coefficients.to_string(index=False))
print(f"\nR-squared del Modello Multiplo: {r_squared_mult:.4f}")

REGRESSIONE LINEARE OLS MULTIPLA (NumPy Algebra)
            Variable  Coefficient (Beta)
           Intercept         -114.260610
            explicit            2.695936
log_artist_followers            2.675881
        release_year            0.058126
  track_duration_min           -0.042135

R-squared del Modello Multiplo: 0.1842
